# Texas Flood Losses

In [1]:
import duckdb
import prep as p

con = duckdb.connect()

p.load_claims(con)

con.sql("SELECT COUNT(*) AS tx_rows FROM claims").df()

,tx_rows
0,394204


In [8]:
table = 'data/tx_claims.parquet'

con.sql('''
            SELECT 
                COUNT(*) as total_rows,
                COUNT(DISTINCT id) as unique_ids, 
                COUNT(*) - count(id) as nulls
            FROM
                'data/tx_claims.parquet'
                
        ''').df()

,total_rows,unique_ids,nulls
0,394204,394204,0


In [2]:
con.sql("DESCRIBE claims").df()

,column_name,column_type,null,key,default,extra
0,id,BIGINT,YES,None,None,None
1,dateOfLoss,TIMESTAMP WITH TIME ZONE,YES,None,None,None
2,yearOfLoss,BIGINT,YES,None,None,None
3,state,VARCHAR,YES,None,None,None
4,countyCode,VARCHAR,YES,None,None,None
5,reportedZipCode,VARCHAR,YES,None,None,None
6,netBuildingPaymentAmount,DOUBLE,YES,None,None,None
7,netContentsPaymentAmount,DOUBLE,YES,None,None,None
8,amountPaidOnBuildingClaim,DOUBLE,YES,None,None,None
9,amountPaidOnContentsClaim,DOUBLE,YES,None,None,None


In [12]:
con.sql('''
        select column_name, null_percentage
        from (SUMMARIZE claims)''').df()

,column_name,null_percentage
0,id,0.00
1,dateOfLoss,0.00
2,yearOfLoss,0.00
3,state,0.00
4,countyCode,0.54
5,reportedZipCode,1.30
6,netBuildingPaymentAmount,0.00
7,netContentsPaymentAmount,0.00
8,amountPaidOnBuildingClaim,18.97
9,amountPaidOnContentsClaim,18.97


In [31]:

''' Paid claim definition: a claim counts as paid when netBuildingPaymentAmount
    or netContentsPaymentAmount is above zero.
    Check: every claim should either have a positive net payment or a reason code
    in nonPaymentReasonBuilding or nonPaymentReasonContents.
    Result: true for nearly all claims. Only 950 rows (about 0.2%) have no reason
    code and no positive payment.
    Note: the amountPaidOnBuildingClaim and amountPaidOnContentsClaim columns are
    not safe for this check. Null values fail every comparison in a CASE, so they
    land in the ELSE branch, which labeled about 68,000 unpaid claims as paid.
'''

con.sql(''' 
        select 
            case 
                when 
                    nonPaymentReasonBuilding is not null
                    or
                    nonPaymentReasonContents is not null
                    or
                    netBuildingPaymentAmount > 0 
                    or
                    netContentsPaymentAmount > 0
                    then 'Explained'
                    else 'Not Explained'
                    end as cases_explained,
             count(*) as "case_count"
        from claims
        group by cases_explained
        ''').df()

,cases_explained,case_count
0,Explained,393254
1,Not Explained,950


In [35]:
con.sql('''
            select 
                count(*) as entry_errors,
            from claims
            where nonPaymentReasonBuilding = '98'
                  or
                  nonPaymentReasonContents = '98'
''').df()

,entry_errors
0,5867


In [42]:
con.sql('''
            select 
                case when nonPaymentReasonBuilding = '98' then 'error' else 'good' end as 'building_code',
                case when netContentsPaymentAmount > 0 then 'payment' else 'no_payment' end as 'contents_payments',
                count(*) as 'count'
            from claims
            where nonPaymentReasonBuilding = '98'
                  or
                  nonPaymentReasonContents = '98'
            group by building_code, contents_payments 
''').df()

,building_code,contents_payments,count
0,good,payment,3
1,error,payment,133
2,good,no_payment,994
3,error,no_payment,4737


In [43]:
con.sql('''
            select 
                case when nonPaymentReasonContents = '98' then 'error' else 'good' end as contents_code,
                case when netBuildingPaymentAmount > 0 then 'payment' else 'no_payment' end as building_payments,
                count(*) as case_count
            from claims
            where nonPaymentReasonBuilding = '98'
                  or
                  nonPaymentReasonContents = '98'
            group by contents_code, building_payments
''').df()

,contents_code,building_payments,case_count
0,good,payment,4
1,error,payment,579
2,error,no_payment,4388
3,good,no_payment,896


In [46]:
con.sql('''
        
            select 
                sum(case when nonPaymentReasonContents = '98' then netBuildingPaymentAmount end) / sum(netBuildingPaymentAmount) as building_money,
                sum(case when nonPaymentReasonBuilding = '98' then netContentsPaymentAmount end) / sum(netContentsPaymentAmount) as Contents_money
            from claims
            
''').df()

,building_money,Contents_money
0,0.000699,0.000269


Dropping 5,867 rows 1.5% of data due to being marked as errors in either contents or building reason codes

In [48]:
con.sql('''
            select 
               count(*)
            from claims
            where nonPaymentReasonContents = '98' or 
                  nonPaymentReasonBuilding = '98'
            
            
''').df()

,count_star()
0,5867


-- countyCode: 2,128 rows are null and 1 row has a non-Texas code (22029, Louisiana).
-- All of these rows are kept, since the state field marks them as Texas claims.
-- The clean sets the non-Texas code to null, so these rows count in Texas totals
-- but not in any county.

In [62]:
con.sql('''
            select 
               count(*) as total,
               count(*) - sum(case when countyCode like '48%' then 1 else 0 end) as non_texas,
               sum(case when countyCode is null then 1 else 0 end) as nulls
            from claims
''').df()

,total,non_texas,nulls
0,394204,2129.0,2128.0


In [67]:
con.sql('''
            select 
               count(*) - sum(case when reportedZipCode like '75%'  or
                                        reportedZipCode like '76%'  or
                                        reportedZipCode like '77%'  or
                                        reportedZipCode like '78%'  or
                                        reportedZipCode like '79%'  or
                                        reportedZipCode like '733%' or
                                        reportedZipCode like '885%'  
                                        then 1 else 0 end) as non_texas,
               sum(case when reportedZipCode is null then 1 else 0 end) as nulls
            from claims
''').df()

,non_texas,nulls
0,5125.0,5116.0


In [68]:
con.sql('''
            select reportedZipCode, count(*) as non_texas_counts
            from claims
            where   reportedZipCode not  like '75%'  and
                    reportedZipCode not  like '76%'  and
                    reportedZipCode not  like '77%'  and
                    reportedZipCode not  like '78%'  and
                    reportedZipCode not  like '79%'  and
                    reportedZipCode not  like '733%' and
                    reportedZipCode not  like '885%' and
                    reportedZipCode is not null
            group by reportedZipCode
            
''').df()

,reportedZipCode,non_texas_counts
0,70000,2
1,73070,1
2,70377,1
3,11111,1
4,72330,1
5,39169,1
6,96799,2


In [69]:
-- reportedZipCode: 5,116 rows are null and 9 rows have non-Texas zips
-- (mostly neighboring states, plus placeholders like 11111).
-- All of these rows are kept, since the state field marks them as Texas claims.
-- The clean sets the non-Texas zips to null, so these rows count in Texas totals
-- but are left out of zip-level results.

SyntaxError: illegal target for annotation (1454185510.py, line 1)